In [21]:
import pandas as pd
import numpy as np
from pathlib import Path

In [22]:
from pathlib import Path
import pandas as pd

# Notebook is inside SIH_162_ML/notebooks/
PROJECT_ROOT = Path.cwd().parent

DATA_FILE = PROJECT_ROOT / "data" / "processed_data" / "firms_cleaned.csv"

print("Project root:")
print(PROJECT_ROOT)

print("\nData file:")
print(DATA_FILE)

print("\nFile exists:", DATA_FILE.exists())

Project root:
c:\Users\DIPANSHU SHUKLA\OneDrive\Desktop\SIH_162_ML

Data file:
c:\Users\DIPANSHU SHUKLA\OneDrive\Desktop\SIH_162_ML\data\processed_data\firms_cleaned.csv

File exists: True


In [23]:
df = pd.read_csv(DATA_FILE)

print("Rows:", len(df))
print("Columns:", df.columns.tolist())

df.head()

Rows: 702968
Columns: ['latitude', 'longitude', 'acq_date', 'acq_time', 'satellite', 'brightness', 'confidence', 'frp', 'daynight', 'version', 'detections_same_location_date', 'source_file']


,latitude,longitude,acq_date,acq_time,satellite,brightness,confidence,frp,daynight,version,detections_same_location_date,source_file
0,8.98492,77.60124,2025-01-01,801,NOAA-20 VIIRS,341.70,n,8.35,D,2,1,DL_FIRE_J1V-C2_812115.zip
1,9.00014,77.61005,2025-01-01,801,NOAA-20 VIIRS,330.68,n,2.68,D,2,1,DL_FIRE_J1V-C2_812115.zip
2,9.27809,78.04165,2025-01-01,801,NOAA-20 VIIRS,330.24,n,1.58,D,2,1,DL_FIRE_J1V-C2_812115.zip
3,9.56098,77.63283,2025-01-01,1949,NOAA-21 VIIRS,303.48,n,0.98,N,2.0NRT,1,DL_FIRE_J2V-C2_812116.zip
4,9.56323,77.63605,2025-01-01,1949,NOAA-21 VIIRS,303.55,n,1.23,N,2.0NRT,1,DL_FIRE_J2V-C2_812116.zip


In [24]:
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd().parent

In [25]:
DATA_FILE = PROJECT_ROOT / "data" / "processed_data" / "firms_cleaned.csv"
EVENT_DIR = PROJECT_ROOT / "data" / "events"

EVENT_DIR.mkdir(parents=True, exist_ok=True)

In [26]:
print(df.shape)
print(df.columns.tolist())

(702968, 12)
['latitude', 'longitude', 'acq_date', 'acq_time', 'satellite', 'brightness', 'confidence', 'frp', 'daynight', 'version', 'detections_same_location_date', 'source_file']


In [27]:
import pandas as pd
import numpy as np
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

DATA_FILE = (
    PROJECT_ROOT
    / "data"
    / "processed_data"
    / "firms_cleaned.csv"
)

df = pd.read_csv(DATA_FILE)

print("Loaded:", len(df), "rows")
print(df.columns.tolist())

Loaded: 702968 rows
['latitude', 'longitude', 'acq_date', 'acq_time', 'satellite', 'brightness', 'confidence', 'frp', 'daynight', 'version', 'detections_same_location_date', 'source_file']


### create timestamps

In [28]:
# Convert acquisition date
df["acq_date"] = pd.to_datetime(
    df["acq_date"],
    errors="coerce"
)

# Make acquisition time a 4-digit string
time_str = (
    pd.to_numeric(df["acq_time"], errors="coerce")
    .fillna(0)
    .astype(int)
    .astype(str)
    .str.zfill(4)
)

# Create timestamp
df["timestamp"] = pd.to_datetime(
    df["acq_date"].dt.strftime("%Y-%m-%d")
    + " "
    + time_str,
    format="%Y-%m-%d %H%M",
    errors="coerce"
)

print("Timestamp created.")
print("Missing timestamps:", df["timestamp"].isna().sum())

df[["acq_date", "acq_time", "timestamp"]].head()

Timestamp created.
Missing timestamps: 0


,acq_date,acq_time,timestamp
0,2025-01-01,801,2025-01-01 08:01:00
1,2025-01-01,801,2025-01-01 08:01:00
2,2025-01-01,801,2025-01-01 08:01:00
3,2025-01-01,1949,2025-01-01 19:49:00
4,2025-01-01,1949,2025-01-01 19:49:00


Remove unusable rows


In [29]:
df = df.dropna(
    subset=[
        "latitude",
        "longitude",
        "timestamp",
        "frp"
    ]
).copy()

df = df.sort_values("timestamp").reset_index(drop=True)

print("Rows after preparation:", len(df))

Rows after preparation: 702968


In [30]:
# ============================================================
# FAST PROTOTYPE EVENT CONSTRUCTION
# ============================================================

SPATIAL_CELL_KM = 1.0
TEMPORAL_WINDOW_MIN = 30

# Approximate conversion
LAT_CELL = SPATIAL_CELL_KM / 111.0

# Longitude conversion depends on latitude.
# Use a conservative approximate cell size.
LON_CELL = SPATIAL_CELL_KM / 100.0

df["lat_cell"] = np.floor(
    df["latitude"] / LAT_CELL
).astype("int32")

df["lon_cell"] = np.floor(
    df["longitude"] / LON_CELL
).astype("int32")

# Sort once
df = df.sort_values(
    ["acq_date", "lat_cell", "lon_cell", "timestamp"]
).reset_index(drop=True)

# Time difference within each spatial cell
df["time_gap"] = (
    df.groupby(["acq_date", "lat_cell", "lon_cell"])["timestamp"]
      .diff()
)

# Start a new event when:
# 1. first observation in cell
# 2. time gap > 30 minutes

df["new_event"] = (
    df["time_gap"].isna()
    |
    (df["time_gap"] > pd.Timedelta(minutes=TEMPORAL_WINDOW_MIN))
)

# Event number inside each spatial cell
df["event_number"] = (
    df.groupby(["acq_date", "lat_cell", "lon_cell"])["new_event"]
      .cumsum()
)

print("Grouping complete.")

Grouping complete.


### create the event table

In [32]:
# ============================================================
# FAST PROTOTYPE EVENT CREATION
# ============================================================

import numpy as np
import pandas as pd

# Make sure data is sorted
df = df.sort_values(
    ["acq_date", "lat_cell", "lon_cell", "event_number"]
).reset_index(drop=True)

# ------------------------------------------------------------
# 1. Numeric event aggregation
# ------------------------------------------------------------

events_df = (
    df.groupby(
        ["acq_date", "lat_cell", "lon_cell", "event_number"],
        sort=False
    )
    .agg(
        start_time=("timestamp", "min"),
        end_time=("timestamp", "max"),
        observation_count=("timestamp", "size"),

        latitude=("latitude", "mean"),
        longitude=("longitude", "mean"),

        mean_frp=("frp", "mean"),
        peak_frp=("frp", "max"),
        total_frp=("frp", "sum"),

        mean_brightness=("brightness", "mean")
    )
    .reset_index()
)

print("Numeric aggregation completed.")
print("Prototype events:", len(events_df))

Numeric aggregation completed.
Prototype events: 439251


### Add persistence

In [33]:
events_df["persistent"] = (
    events_df["observation_count"] >= 2
)

print(
    events_df["persistent"].value_counts()
)

persistent
False    305200
True     134051
Name: count, dtype: int64


save

In [34]:
EVENT_DIR = PROJECT_ROOT / "data" / "events"
EVENT_DIR.mkdir(parents=True, exist_ok=True)

EVENT_FILE = EVENT_DIR / "thermal_events_prototype.csv"

events_df.to_csv(
    EVENT_FILE,
    index=False
)

print("Saved successfully:")
print(EVENT_FILE)

Saved successfully:
c:\Users\DIPANSHU SHUKLA\OneDrive\Desktop\SIH_162_ML\data\events\thermal_events_prototype.csv
